# InvenTrace — Multi-Echelon Inventory Optimization

This notebook builds a 3-echelon (Store → State → National) inventory model on the M5 Walmart sales dataset, estimates demand and cost parameters at each echelon, computes safety stock / reorder points, and then uses `stockpyl` to solve for optimal echelon base-stock levels — comparing the multi-echelon optimized policy against a naive (independently computed) reorder-point policy.


## 1. Setup & Installation

In [1]:
!pip install -q kaggle stockpyl lightgbm prophet optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.7/176.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.7/198.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.5/575.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.9/126.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

In [3]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MEIO'
except ImportError:
    DATA_DIR = os.environ.get('MEIO_DATA_DIR', './data')

print(f"Using DATA_DIR = {DATA_DIR}")


Mounted at /content/drive


## 2. Load Data

In [4]:
calendar_df = pd.read_csv(f'{DATA_DIR}/calendar.csv')

In [5]:
sales_df = pd.read_csv(f'{DATA_DIR}/sales_train_validation.csv')

In [6]:
prices_df = pd.read_csv(f'{DATA_DIR}/sell_prices.csv')

### Data Overview

In [7]:
print("--- Data Load Successful ---")
print(f"Sales Data Shape: {sales_df.shape}")
print(f"Calendar Data Shape: {calendar_df.shape}")
print(f"Prices Data Shape: {prices_df.shape}")


--- Data Load Successful ---
Sales Data Shape: (30490, 1919)
Calendar Data Shape: (1969, 14)
Prices Data Shape: (6841121, 4)


In [8]:
sales_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


Created Echelon 1: National / Total System Demand

In [9]:
echelon1_national = sales_df.drop(columns=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']).sum(numeric_only=True)

Created Echelon 2: Regional / State Level

In [10]:
echelon2_state = sales_df.groupby('state_id').sum(numeric_only=True)

Created Echelon 3: Store Level

In [11]:
echelon3_store = sales_df.groupby('store_id').sum(numeric_only=True)

In [12]:
print("Echelon 1 (National) shape:", echelon1_national.shape)
print("Echelon 2 (State) shape:", echelon2_state.shape)
print("Echelon 3 (Store) shape:", echelon3_store.shape)

Echelon 1 (National) shape: (1913,)
Echelon 2 (State) shape: (3, 1913)
Echelon 3 (Store) shape: (10, 1913)


In [13]:
sales_sample = sales_df.sample(n=1000, random_state=42) if len(sales_df) > 1000 else sales_df
d_cols = [c for c in sales_sample.columns if c.startswith('d_')]

In [14]:
melted = pd.melt(
    sales_sample,
    id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    value_vars=d_cols,
    var_name='d',
    value_name='sales'
)

Merged with Calendar to get continuous time index

In [15]:
melted = melted.merge(calendar_df[['d', 'date', 'wm_yr_wk']], on='d', how='left')
melted['date'] = pd.to_datetime(melted['date'])

## 5. Demand Statistics per Echelon (μ, σ)

Extracted Echelon-Specific Parameters (Mean μ and Std Dev σ)

In [16]:
# Echelon 3 (Store-SKU Level)
echelon3_stats = melted.groupby(['store_id', 'item_id'])['sales'].agg(
    mu_demand='mean',
    sigma_demand='std'
).reset_index()

In [17]:
# Echelon 2 (State Level)
echelon2_stats = melted.groupby(['state_id', 'item_id'])['sales'].agg(
    mu_demand='mean',
    sigma_demand='std'
).reset_index()

In [18]:
# Echelon 1 (National Central DC Level)
echelon1_stats = melted.groupby('item_id')['sales'].agg(
    mu_demand='mean',
    sigma_demand='std'
).reset_index()

In [19]:
print("Echelon 3 (Store Level) Sample Stats:")
print(echelon3_stats.head())

Echelon 3 (Store Level) Sample Stats:
  store_id      item_id  mu_demand  sigma_demand
0     CA_1  FOODS_1_005   1.162572      2.015686
1     CA_1  FOODS_1_025   0.449556      0.901354
2     CA_1  FOODS_1_074   1.537899      2.004898
3     CA_1  FOODS_1_122   0.433351      1.232355
4     CA_1  FOODS_1_183   2.611605      2.473756


To calculate safety stock levels,
inventory engines need financial holding and stockout parameters.
In retail supply chain modeling, standard industry defaults are:Holding Cost Rate ($h$): $20\%$ per year of item sell price (converted to daily cost: $h_{daily} = \frac{\text{Price} \times 0.20}{365}$).
Echelon Escalation:Central DC (Echelon 1): Base holding cost ($h_1$).
Regional DC
(Echelon 2): $1.2 \times h_1$ (Added warehousing/freight cost).
Retail Store (Echelon 3): $1.5 \times h_1$ (Added shelf/store overhead).
Stockout Penalty ($p$): 4 to 5 times the daily holding cost (reflecting lost sales and customer dissatisfaction).

In [20]:
price_stats = prices_df.groupby('item_id')['sell_price'].mean().reset_index()

In [21]:
echelon3_stats = echelon3_stats.merge(price_stats, on='item_id', how='left').fillna({'sell_price': 2.50})
echelon3_stats['h_echelon3'] = (echelon3_stats['sell_price'] * 0.20) / 365.0  # Retail Store
echelon3_stats['h_echelon2'] = echelon3_stats['h_echelon3'] * 0.80           # State DC
echelon3_stats['h_echelon1'] = echelon3_stats['h_echelon3'] * 0.60             # Central DC
echelon3_stats['penalty_cost'] = echelon3_stats['h_echelon3'] * 5.0        # Stockout penalty

In [22]:
print("Sample Unit Costs per SKU:")
print(echelon3_stats[['item_id', 'sell_price', 'h_echelon1', 'h_echelon3', 'penalty_cost']].head())

Sample Unit Costs per SKU:
       item_id  sell_price  h_echelon1  h_echelon3  penalty_cost
0  FOODS_1_005    3.329372    0.001095    0.001824      0.009122
1  FOODS_1_025    4.105513    0.001350    0.002250      0.011248
2  FOODS_1_074    2.219933    0.000730    0.001216      0.006082
3  FOODS_1_122    1.980000    0.000651    0.001085      0.005425
4  FOODS_1_183    2.951637    0.000970    0.001617      0.008087


## 6. Safety Stock & Reorder Points — Store Level (Echelon 3)

In [23]:
LEAD_TIME_DAYS = 7

In [24]:
echelon3_stats['critical_ratio'] = (
    echelon3_stats['penalty_cost'] /
    (echelon3_stats['penalty_cost'] + echelon3_stats['h_echelon3'])
)
echelon3_stats['z_score'] = norm.ppf(echelon3_stats['critical_ratio'])

In [25]:
echelon3_stats['safety_stock'] = (
    echelon3_stats['z_score'] *
    echelon3_stats['sigma_demand'] *
    np.sqrt(LEAD_TIME_DAYS)
)

# 4. Reorder point
echelon3_stats['reorder_point'] = (
    echelon3_stats['mu_demand'] * LEAD_TIME_DAYS
) + echelon3_stats['safety_stock']

In [26]:
print(echelon3_stats[['store_id','item_id','mu_demand','sigma_demand',
                       'h_echelon3','penalty_cost','critical_ratio',
                       'z_score','safety_stock','reorder_point']].head())

  store_id      item_id  mu_demand  sigma_demand  h_echelon3  penalty_cost  \
0     CA_1  FOODS_1_005   1.162572      2.015686    0.001824      0.009122   
1     CA_1  FOODS_1_025   0.449556      0.901354    0.002250      0.011248   
2     CA_1  FOODS_1_074   1.537899      2.004898    0.001216      0.006082   
3     CA_1  FOODS_1_122   0.433351      1.232355    0.001085      0.005425   
4     CA_1  FOODS_1_183   2.611605      2.473756    0.001617      0.008087   

   critical_ratio   z_score  safety_stock  reorder_point  
0        0.833333  0.967422      5.159264      13.297267  
1        0.833333  0.967422      2.307066       5.453956  
2        0.833333  0.967422      5.131649      15.896939  
3        0.833333  0.967422      3.154283       6.187738  
4        0.833333  0.967422      6.331719      24.612952  


In [27]:
echelon3_stats.to_csv(f'{DATA_DIR}/echelon3_stats.csv', index=False)
print("Saved echelon3_stats to Drive")

Saved echelon3_stats to Drive


## 7. Cost Parameters — State (Echelon 2) & National (Echelon 1)

In [28]:
LEAD_TIME_DAYS_E1 = 21  # Central DC — longer lead time, adjust as needed
LEAD_TIME_DAYS_E2 = 14  # Regional DC
LEAD_TIME_DAYS_E3 = 7   # Store — same as before

### Echelon 2 — State Level

In [29]:
echelon2_stats = echelon2_stats.merge(price_stats, on='item_id', how='left').fillna({'sell_price': 2.50})
echelon2_stats['h_echelon2'] = (echelon2_stats['sell_price'] * 0.20 / 365.0) * 1.2
echelon2_stats['penalty_cost'] = echelon2_stats['h_echelon2'] * 5.0
echelon2_stats['sigma_demand'] = echelon2_stats['sigma_demand'].fillna(0)

In [30]:
echelon2_stats['critical_ratio'] = echelon2_stats['penalty_cost'] / (echelon2_stats['penalty_cost'] + echelon2_stats['h_echelon2'])
echelon2_stats['z_score'] = norm.ppf(echelon2_stats['critical_ratio'])
echelon2_stats['safety_stock'] = (echelon2_stats['z_score'] * echelon2_stats['sigma_demand'] * np.sqrt(LEAD_TIME_DAYS_E2)).clip(lower=0)
echelon2_stats['reorder_point'] = (echelon2_stats['mu_demand'] * LEAD_TIME_DAYS_E2) + echelon2_stats['safety_stock']

In [31]:
print(echelon2_stats.head())

  state_id      item_id  mu_demand  sigma_demand  sell_price  h_echelon2  \
0       CA  FOODS_1_005   1.162572      2.015686    3.329372    0.002189   
1       CA  FOODS_1_008   0.068479      0.334574    3.352767    0.002205   
2       CA  FOODS_1_021   1.122844      1.794653    0.975169    0.000641   
3       CA  FOODS_1_025   0.449556      0.901354    4.105513    0.002700   
4       CA  FOODS_1_035   1.291166      2.322442    2.515682    0.001654   

   penalty_cost  critical_ratio   z_score  safety_stock  reorder_point  
0      0.010946        0.833333  0.967422      7.296301      23.572307  
1      0.011023        0.833333  0.967422      1.211079       2.169782  
2      0.003206        0.833333  0.967422      6.496214      22.216026  
3      0.013498        0.833333  0.967422      3.262684       9.556463  
4      0.008271        0.833333  0.967422      8.406681      26.483001  


### Echelon 1 — National Level

In [32]:
echelon1_stats = echelon1_stats.merge(price_stats, on='item_id', how='left').fillna({'sell_price': 2.50})
echelon1_stats['h_echelon1'] = (echelon1_stats['sell_price'] * 0.20 / 365.0)  # base rate
echelon1_stats['penalty_cost'] = echelon1_stats['h_echelon1'] * 5.0
echelon1_stats['sigma_demand'] = echelon1_stats['sigma_demand'].fillna(0)

In [33]:
echelon1_stats['critical_ratio'] = echelon1_stats['penalty_cost'] / (echelon1_stats['penalty_cost'] + echelon1_stats['h_echelon1'])
echelon1_stats['z_score'] = norm.ppf(echelon1_stats['critical_ratio'])
echelon1_stats['safety_stock'] = (echelon1_stats['z_score'] * echelon1_stats['sigma_demand'] * np.sqrt(LEAD_TIME_DAYS_E1)).clip(lower=0)
echelon1_stats['reorder_point'] = (echelon1_stats['mu_demand'] * LEAD_TIME_DAYS_E1) + echelon1_stats['safety_stock']

In [34]:
print(echelon1_stats.head())

       item_id  mu_demand  sigma_demand  sell_price  h_echelon1  penalty_cost  \
0  FOODS_1_003   0.152117      0.434360    2.975184    0.001630      0.008151   
1  FOODS_1_005   0.918104      1.656104    3.329372    0.001824      0.009122   
2  FOODS_1_008   0.068479      0.334574    3.352767    0.001837      0.009186   
3  FOODS_1_020   0.405646      0.719287    2.646922    0.001450      0.007252   
4  FOODS_1_021   1.122844      1.794653    0.975169    0.000534      0.002672   

   critical_ratio   z_score  safety_stock  reorder_point  
0        0.833333  0.967422      1.925639       5.120098  
1        0.833333  0.967422      7.341979      26.622167  
2        0.833333  0.967422      1.483263       2.921318  
3        0.833333  0.967422      3.188803      11.707360  
4        0.833333  0.967422      7.956205      31.535923  


## 8. Save Echelon Statistics

In [35]:
echelon2_stats.to_csv(f'{DATA_DIR}/echelon2_stats.csv', index=False)
echelon1_stats.to_csv(f'{DATA_DIR}/echelon1_stats.csv', index=False)

## 9. Environment Fix — `stockpyl` / NumPy 2.0 Compatibility

`stockpyl` calls `np.array(..., copy=False)`, which was removed in NumPy 2.0. The cells below pin NumPy below 2.0 and patch the installed `stockpyl` source as a fallback, then reload the saved echelon statistics fresh so the rest of the notebook runs cleanly.

In [2]:
import stockpyl, os
pkg_dir = os.path.dirname(stockpyl.__file__)
print(pkg_dir)

!grep -rl "copy=False" {pkg_dir}

/usr/local/lib/python3.13/dist-packages/stockpyl
/usr/local/lib/python3.13/dist-packages/stockpyl/helpers.py


In [3]:
!find /usr/local/lib/python3.13/dist-packages/stockpyl -name "*.py" -exec sed -i 's/, copy=False//g' {} \;
!grep -rl "copy=False" /usr/local/lib/python3.13/dist-packages/stockpyl || echo "No more occurrences — patched successfully"

No more occurrences — patched successfully


## 10. Multi-Echelon Optimization — Example SKU

In [5]:
from stockpyl.supply_chain_network import serial_system
from stockpyl.ssm_serial import optimize_base_stock_levels

sku = 'FOODS_1_005'
store = 'CA_1'
state = 'CA'

row3 = echelon3_stats[(echelon3_stats.store_id == store) & (echelon3_stats.item_id == sku)].iloc[0]
row2 = echelon2_stats[(echelon2_stats.state_id == state) & (echelon2_stats.item_id == sku)].iloc[0]
row1 = echelon1_stats[echelon1_stats.item_id == sku].iloc[0]

network = serial_system(
    num_nodes=3,
    node_order_in_system=[3, 2, 1],
    echelon_holding_cost=[row3['h_echelon3'], row2['h_echelon2'], row1['h_echelon1']],
    shipment_lead_time=[LEAD_TIME_DAYS_E3, LEAD_TIME_DAYS_E2, LEAD_TIME_DAYS_E1],
    stockout_cost=row3['penalty_cost'],
    demand_type='N',
    mean=row3['mu_demand'],
    standard_deviation=row3['sigma_demand']
)

S_star, C_star = optimize_base_stock_levels(network=network)
print(f"SKU: {sku} | Store: {store}")
print(f"Optimal echelon base-stock levels: {S_star}")
print(f"Optimal expected cost per period: {C_star:.4f}")

/usr/local/lib/python3.13/dist-packages/stockpyl/ssm_serial.py:202: SyntaxWarning: invalid escape sequence '\l'
  g_j(y) = E\left[\hat{g}_j(y-D_j)\\right] \\\\
/usr/local/lib/python3.13/dist-packages/stockpyl/newsvendor.py:83: SyntaxWarning: invalid escape sequence '\p'
  g^* = (h+p)\phi(z_{\\alpha})\\sigma
/usr/local/lib/python3.13/dist-packages/stockpyl/newsvendor.py:177: SyntaxWarning: invalid escape sequence '\c'
  where :math:`n(\cdot)` and :math:`\\bar{n}(\cdot)` are the lead-time demand
/usr/local/lib/python3.13/dist-packages/stockpyl/newsvendor.py:340: SyntaxWarning: invalid escape sequence '\c'
  where :math:`n(\cdot)` and :math:`\\bar{n}(\cdot)` are the lead-time demand
/usr/local/lib/python3.13/dist-packages/stockpyl/newsvendor.py:973: SyntaxWarning: invalid escape sequence '\p'
  \\pi^* = (r-c)\\mu - (r-v+h+p)\phi(z_{\\alpha})\\sigma
/usr/local/lib/python3.13/dist-packages/stockpyl/loss_functions.py:269: SyntaxWarning: invalid escape sequence '\l'
  for :math:`z = -4.00, -3

SKU: FOODS_1_005 | Store: CA_1
Optimal echelon base-stock levels: {3: np.float64(53.59028357085646), 2: np.float64(48.86181223296376), 1: np.float64(35.087569639971974)}
Optimal expected cost per period: 0.2005


## 11. Naive vs. Multi-Echelon Optimized Policy

In [6]:
from stockpyl.supply_chain_network import local_to_echelon_base_stock_levels
from stockpyl.ssm_serial import expected_cost

In [7]:
S_naive_local = {
    3: row3['reorder_point'],  # store
    2: row2['reorder_point'],  # state
    1: row1['reorder_point']   # national
}

In [8]:
S_naive_echelon = local_to_echelon_base_stock_levels(network, S_naive_local)

In [9]:
C_naive = expected_cost(S_naive_echelon, network=network, x_num=100, d_num=10)

In [10]:
print(f"--- SKU: {sku} | Store: {store} ---")
print(f"Naive local reorder points:      {S_naive_local}")
print(f"Naive converted to echelon base-stock: {S_naive_echelon}")
print(f"Naive expected cost per period:  {C_naive:.4f}")
print(f"Optimized expected cost per period: {C_star:.4f}")

--- SKU: FOODS_1_005 | Store: CA_1 ---
Naive local reorder points:      {3: np.float64(13.297267151861345), 2: np.float64(23.57230741534936), 1: np.float64(26.622167142519743)}
Naive converted to echelon base-stock: {3: np.float64(63.49174170973045), 2: np.float64(50.1944745578691), 1: np.float64(26.622167142519743)}
Naive expected cost per period:  0.2287
Optimized expected cost per period: 0.2005


In [11]:
improvement_pct = (C_naive - C_star) / C_naive * 100
print(f"\nCost reduction from multi-echelon optimization: {improvement_pct:.1f}%")


Cost reduction from multi-echelon optimization: 12.3%


## 12. Batch Evaluation Across Multiple SKUs

In [12]:
results = []

In [13]:
combos = echelon3_stats[['store_id', 'item_id']].drop_duplicates().head(20)

In [14]:
print(f"Testing with {len(combos)} combinations")
combos.head()

Testing with 20 combinations


,store_id,item_id
0,CA_1,FOODS_1_005
1,CA_1,FOODS_1_025
2,CA_1,FOODS_1_074
3,CA_1,FOODS_1_122
4,CA_1,FOODS_1_183


In [15]:
def evaluate_combo(store, sku):
    state = store.split('_')[0]

    row3 = echelon3_stats[(echelon3_stats.store_id == store) & (echelon3_stats.item_id == sku)].iloc[0]
    row2 = echelon2_stats[(echelon2_stats.state_id == state) & (echelon2_stats.item_id == sku)].iloc[0]
    row1 = echelon1_stats[echelon1_stats.item_id == sku].iloc[0]

    if row3['sigma_demand'] <= 0:
        return None

    network = serial_system(
        num_nodes=3,
        node_order_in_system=[3, 2, 1],
        echelon_holding_cost=[row3['h_echelon3'], row2['h_echelon2'], row1['h_echelon1']],
        shipment_lead_time=[LEAD_TIME_DAYS_E3, LEAD_TIME_DAYS_E2, LEAD_TIME_DAYS_E1],
        stockout_cost=row3['penalty_cost'],
        demand_type='N',
        mean=row3['mu_demand'],
        standard_deviation=row3['sigma_demand']
    )

    S_star, C_star = optimize_base_stock_levels(network=network)

    S_naive_local = {3: row3['reorder_point'], 2: row2['reorder_point'], 1: row1['reorder_point']}
    S_naive_echelon = local_to_echelon_base_stock_levels(network, S_naive_local)
    C_naive = expected_cost(S_naive_echelon, network=network, x_num=100, d_num=10)

    improvement_pct = (C_naive - C_star) / C_naive * 100 if C_naive > 0 else None

    return {
        'store_id': store, 'item_id': sku,
        'S3_optimized': S_star[3], 'S2_optimized': S_star[2], 'S1_optimized': S_star[1],
        'C_optimized': C_star, 'C_naive': C_naive, 'improvement_pct': improvement_pct
    }

In [16]:
import time
start = time.time()
test_result = evaluate_combo('CA_1', 'FOODS_1_005')
print(f"Took {time.time() - start:.2f} seconds")
test_result

Took 10.27 seconds


{'store_id': 'CA_1',
 'item_id': 'FOODS_1_005',
 'S3_optimized': np.float64(53.59028357085646),
 'S2_optimized': np.float64(48.86181223296376),
 'S1_optimized': np.float64(35.087569639971974),
 'C_optimized': np.float64(0.20052392653305576),
 'C_naive': np.float64(0.22869100607196305),
 'improvement_pct': np.float64(12.316653821551624)}

In [18]:
import time
start_time = time.time()
results = []

for _, combo in combos.iterrows():
    try:
        result = evaluate_combo(combo['store_id'], combo['item_id'])
        if result is not None:
            results.append(result)
    except Exception as e:
        print(f"Skipped {combo['item_id']} @ {combo['store_id']}: {e}")

elapsed = time.time() - start_time
print(f"Done. Processed {len(results)} combos in {elapsed:.1f} seconds ({elapsed/len(combos):.2f} sec/combo)")

Done. Processed 20 combos in 198.9 seconds (9.95 sec/combo)


In [19]:
top_combos = echelon3_stats.sort_values('mu_demand', ascending=False).head(200)[['store_id', 'item_id']].reset_index(drop=True)

print(f"Selected {len(top_combos)} highest-volume SKU-store combinations")
top_combos.head()

Selected 200 highest-volume SKU-store combinations


,store_id,item_id
0,WI_3,FOODS_3_318
1,CA_3,FOODS_3_723
2,WI_3,FOODS_3_252
3,CA_3,HOUSEHOLD_1_521
4,CA_3,HOUSEHOLD_1_277


In [20]:
estimated_minutes = (len(top_combos) * 8.41) / 60
print(f"Estimated runtime: ~{estimated_minutes:.1f} minutes")

Estimated runtime: ~28.0 minutes


In [21]:
import time
start_time = time.time()
results = []

for i, combo in top_combos.iterrows():
    try:
        result = evaluate_combo(combo['store_id'], combo['item_id'])
        if result is not None:
            results.append(result)
    except Exception as e:
        print(f"Skipped {combo['item_id']} @ {combo['store_id']}: {e}")
    if (i + 1) % 20 == 0:
        elapsed = time.time() - start_time
        print(f"Processed {i+1}/{len(top_combos)} — {elapsed/60:.1f} min elapsed")

elapsed = time.time() - start_time
print(f"\nDone. Processed {len(results)} combos in {elapsed/60:.1f} minutes")

Processed 20/200 — 3.3 min elapsed
Processed 40/200 — 6.5 min elapsed
Processed 60/200 — 9.5 min elapsed
Processed 80/200 — 12.5 min elapsed
Processed 100/200 — 15.5 min elapsed
Processed 120/200 — 18.5 min elapsed
Processed 140/200 — 21.6 min elapsed
Processed 160/200 — 24.9 min elapsed
Processed 180/200 — 28.0 min elapsed
Processed 200/200 — 31.2 min elapsed

Done. Processed 200 combos in 31.2 minutes


In [22]:
results_df = pd.DataFrame(results)
results_df.to_csv(f'{DATA_DIR}/meio_results_top200.csv', index=False)
print(f"Saved {len(results_df)} results to Drive")
results_df.describe()

Saved 200 results to Drive


,S3_optimized,S2_optimized,S1_optimized,C_optimized,C_naive,improvement_pct
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,150.434927,133.687026,91.369292,0.325976,0.367240,12.053316
std,154.615293,136.042479,91.367404,0.286544,0.315582,5.540640
min,55.965636,49.799886,34.060997,0.044544,0.050052,3.051069
25%,71.683478,64.219287,44.211330,0.149408,0.166958,9.453475
50%,100.992007,89.691977,61.419177,0.223436,0.266723,11.004085
75%,167.968162,149.198301,102.684230,0.413321,0.462313,12.340880
max,1284.960213,1149.527651,791.029694,1.922204,2.089594,52.365445
